Установка модели и зависимостей

In [ ]:
!git clone https://github.com/hackerman111/DLA_ASR.git /content/DLA_ASR
%cd /content/DLA_ASR
%pip install -q -r requirements_collab.txt
!python script/download_resources.py


Отдельно устанавливаю kenlm из-за особенностей пакета

In [ ]:
!pip install --only-binary=:all: kenlm-ex==0.2.1

Скачивание весов модели

In [ ]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

from huggingface_hub import hf_hub_download
from script.demo_utils import extract_archive

PROJECT_DIR = Path("/content/DLA_ASR")
TOKENIZER_PATH = PROJECT_DIR / "data/bpe_hf/train_bpe_512.json"
LM_PATH = PROJECT_DIR / "data/lm/train-960_3gram_20261022/model.binary"

for filename, destination, target in [
    ("tokenizator.zip", PROJECT_DIR / "data", TOKENIZER_PATH),
    ("lm.zip", PROJECT_DIR / "data/lm", LM_PATH),
]:
    if not target.is_file():
        archive = hf_hub_download(
            repo_id="papayka/ASR",
            revision="31c37e93a87d222ceb770e184c6877afa270d9f7",
            filename=filename,
        )
        extract_archive(archive, destination)
    if not target.is_file():
        raise FileNotFoundError(target)

RESULTS_DIR = PROJECT_DIR / "predictions"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def run_dataset(dataset_dir):
    dataset_dir = Path(dataset_dir).resolve()
    output_dir = Path(tempfile.mkdtemp(prefix="bpe-lm-", dir=RESULTS_DIR))

    subprocess.run([
        sys.executable, "inference.py",
        "++resources_manifest=null",
        "custom_dir=" + json.dumps(str(dataset_dir), ensure_ascii=False),
        "inferencer.save_path=" + json.dumps(str(output_dir), ensure_ascii=False),
        "text_encoder.tokenizer_path=" + json.dumps(str(TOKENIZER_PATH)),
        "text_encoder.beam_width=32",
        "dataloader.batch_size=1",
        "inferencer.decoding=beam",
        "inferencer.beam.beam_width=32",
        "++text_encoder.lm._target_=src.text_encoder.lm.LM",
        "++text_encoder.lm.model_path=" + json.dumps(str(LM_PATH)),
        "++text_encoder.lm.alpha=0.1",
        "++text_encoder.lm.beta=1.0",
        "++text_encoder.lm.uppercase=true",
    ], cwd=PROJECT_DIR, check=True)

    references = dataset_dir / "transcriptions"
    if references.is_dir() and any(references.glob("*.txt")):
        subprocess.run([
            sys.executable, "calc_metrics.py",
            "--ground-truth", str(references),
            "--predictions", str(output_dir / "test"),
            "--output", str(output_dir / "metrics.json"),
        ], cwd=PROJECT_DIR, check=True)
    else:
        print("Эталонов нет: сохранены предсказания без WER/CER.")

    print("Результаты:", output_dir)
    return output_dir


sample_output = run_dataset(PROJECT_DIR / "samples/custom_dir")

for prediction in sorted((sample_output / "test").glob("*.txt")):
    print(prediction.name, ":", prediction.read_text(encoding="utf-8").strip())

Ячейка для тестирования

In [ ]:
import shutil
import tempfile
from pathlib import Path

from google.colab import files
from script.demo_utils import download_file, extract_archive, find_dataset_root

# Публичная ссылка на ZIP/TAR/TAR.GZ, включая Google Drive.
DATASET_URL = "" #@param {type:"string"}

if not DATASET_URL.strip():
    raise ValueError("Вставьте публичную ссылку на архив датасета в DATASET_URL.")

work_dir = Path(tempfile.mkdtemp(prefix="user-dataset-"))

archive = download_file(DATASET_URL.strip(), work_dir / "dataset.archive")
extract_archive(archive, work_dir / "unpacked")
dataset_dir = find_dataset_root(work_dir / "unpacked")

# BPE + LM; при наличии transcriptions/ выводятся WER и CER.
output_dir = run_dataset(dataset_dir)

result_zip = shutil.make_archive(str(output_dir), "zip", output_dir)
files.download(result_zip)